### Import Required Libraries and Set Up Environment Variables

In [12]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [24]:
# Create a .env file with your NASA API key directly
nasa_api_key = "qZNll3oTgpkCsIoY5XoRvQj3ecb6hX3UwOcaSkHI"  # Note the quotes around the key

# Create the .env file
with open('.env', 'w') as f:
    f.write(f"NASA_API_KEY={nasa_api_key}")

print("Created .env file")
print(".env file exists:", os.path.exists('.env'))

# Now reload the environment variables
from dotenv import load_dotenv
load_dotenv(override=True)
NASA_API_KEY = os.getenv('NASA_API_KEY')
print("API key loaded after creating .env:", "Yes" if NASA_API_KEY else "No")

Created .env file
.env file exists: True
API key loaded after creating .env: Yes


In [19]:
# Dependencies
import requests
import time
from dotenv import load_dotenv
import os
import pandas as pd
import json
import os
from datetime import datetime
## Load the NASA_API_KEY from the env file
load_dotenv()
NASA_API_KEY = os.getenv('NASA_API_KEY')

### CME Data

In [20]:
# Set the base URL to NASA's DONKI API:
base_url = "https://api.nasa.gov/DONKI/"

# Set the specifier for CMEs:
CME = "CME"

# Search for CMEs published between a begin and end date
startDate = "2013-05-01"
endDate   = "2024-05-01"

# Build URL for CME
query_url_CME = f"{base_url}{CME}?startDate={startDate}&endDate={endDate}&api_key={NASA_API_KEY}"

In [21]:
# Make a "GET" request for the CME URL and store it in a variable named cme_response
cme_response = requests.get(query_url_CME)

In [22]:
# Convert the response variable to json and store it as a variable named cme_json
cme_json = cme_response.json()

In [23]:
# Preview ONLY the first element from the cme_json list you created in JSON format
# Do NOT print out the entire list
# Use json.dumps with argument indent=4 to format data

print(json.dumps(cme_json[0], indent=4))

{
    "activityID": "2013-05-01T03:12:00-CME-001",
    "catalog": "M2M_CATALOG",
    "startTime": "2013-05-01T03:12Z",
    "instruments": [
        {
            "displayName": "SOHO: LASCO/C2"
        },
        {
            "displayName": "SOHO: LASCO/C3"
        },
        {
            "displayName": "STEREO A: SECCHI/COR2"
        },
        {
            "displayName": "STEREO B: SECCHI/COR2"
        }
    ],
    "sourceLocation": "",
    "activeRegionNum": null,
    "note": "",
    "submissionTime": "2013-08-07T16:54Z",
    "versionId": 1,
    "link": "https://webtools.ccmc.gsfc.nasa.gov/DONKI/view/CME/2349/-1",
    "cmeAnalyses": [
        {
            "isMostAccurate": true,
            "time21_5": "2013-05-01T07:07Z",
            "latitude": 12.0,
            "longitude": -120.0,
            "halfAngle": 36.0,
            "speed": 860.0,
            "type": "C",
            "featureCode": "null",
            "imageType": null,
            "measurementTechnique": "null",
   

In [35]:
# Convert cme_json to a Pandas DataFrame 
cme = pd.DataFrame(cme_json)
cme.head()
# Keep only the columns: activityID, startTime, linkedEvents
cme = cme[['activityID', 'startTime', 'linkedEvents']]
cme.head()

,activityID,startTime,linkedEvents
0,2013-05-01T03:12:00-CME-001,2013-05-01T03:12Z,[{'activityID': '2013-05-04T04:52:00-IPS-001'}]
1,2013-05-02T05:24:00-CME-001,2013-05-02T05:24Z,None
2,2013-05-02T14:36:00-CME-001,2013-05-02T14:36Z,None
3,2013-05-03T18:00:00-CME-001,2013-05-03T18:00Z,None
4,2013-05-03T22:36:00-CME-001,2013-05-03T22:36Z,[{'activityID': '2013-05-07T04:37:00-IPS-001'}]


In [36]:
# Notice that the linkedEvents column allows us to identify the corresponding GST
# Remove rows with missing 'linkedEvents' since we won't be able to assign these to GSTs
cme = cme[cme['linkedEvents'].notna()]

display(cme.head())


,activityID,startTime,linkedEvents
0,2013-05-01T03:12:00-CME-001,2013-05-01T03:12Z,[{'activityID': '2013-05-04T04:52:00-IPS-001'}]
4,2013-05-03T22:36:00-CME-001,2013-05-03T22:36Z,[{'activityID': '2013-05-07T04:37:00-IPS-001'}]
7,2013-05-09T19:29:00-CME-001,2013-05-09T19:29Z,[{'activityID': '2013-05-12T23:30:00-IPS-001'}]
10,2013-05-13T02:54:00-CME-001,2013-05-13T02:54Z,[{'activityID': '2013-05-13T01:53:00-FLR-001'}...
13,2013-05-13T16:18:00-CME-001,2013-05-13T16:18Z,[{'activityID': '2013-05-13T15:40:00-FLR-001'}...


In [38]:
# Notice that the linkedEvents sometimes contains multiple events per row
# Write a nested for loop that iterates first over each row in the cme DataFrame (using the index)
# and then iterates over the values in 'linkedEvents' 
# and adds the elements individually to a list of dictionaries where each row is one element 

# Initialize an empty list to store the expanded rows
expanded_rows = []

# Iterate over each index in the DataFrame
for i in cme.index:
    activityID = cme.loc[i, 'activityID']
    startTime = cme.loc[i, 'startTime']
    linkedEvents = cme.loc[i, 'linkedEvents']
    # Iterate over each dictionary in the list
    for item in linkedEvents:
        # Append a new dictionary to the expanded_rows list for each dictionary item and corresponding 'activityID' and 'startTime' value
      expanded_rows.append({
            'activityID': activityID,
            'startTime': startTime,
            'linkedEvents': item
        })
# Create a new DataFrame from the expanded rows
cme = pd.DataFrame(expanded_rows)
display(cme.head())

,activityID,startTime,linkedEvents
0,2013-05-01T03:12:00-CME-001,2013-05-01T03:12Z,activityID
1,2013-05-03T22:36:00-CME-001,2013-05-03T22:36Z,activityID
2,2013-05-09T19:29:00-CME-001,2013-05-09T19:29Z,activityID
3,2013-05-13T02:54:00-CME-001,2013-05-13T02:54Z,activityID
4,2013-05-13T02:54:00-CME-001,2013-05-13T02:54Z,activityID


In [48]:
# Create a function called extract_activityID_from_dict that takes a dict as input such as in linkedEvents
# and verify below that it works as expected using one row from linkedEvents as an example
# Be sure to use a try and except block to handle errors
# Updated function to handle string values properly
def extract_activityID_from_dict(input_dict):
    try:
        # If it's a dictionary with 'activityID' key
        if isinstance(input_dict, dict) and 'activityID' in input_dict:
            return input_dict['activityID']
        
        # Handle case where it might already be a string containing the ID
        elif isinstance(input_dict, str):
            # Check if it looks like an ID (contains date and type format)
            if 'T' in input_dict and '-' in input_dict and ':' in input_dict:
                return input_dict
            else:
                return None
        else:
            return None
    except (ValueError, TypeError, KeyError) as e:
        # Log the error or print it for debugging
        print(f"Error extracting activityID: {e}")
        return None

# Test the function with the data from cme_json directly
if isinstance(cme_json, list) and len(cme_json) > 0 and 'linkedEvents' in cme_json[0]:
    # Get the first linkedEvent from the first CME
    linked_event = cme_json[0]['linkedEvents'][0]
    
    # Test the function
    result = extract_activityID_from_dict(linked_event)
    
    # Display the result
    print("Sample linkedEvent:", linked_event)
    print("Extracted activityID:", result)


Sample linkedEvent: {'activityID': '2013-05-04T04:52:00-IPS-001'}
Extracted activityID: 2013-05-04T04:52:00-IPS-001


In [50]:
# Apply this function to each row in the 'linkedEvents' column (you can use apply() and a lambda function)
# and create a new column called 'GST_ActivityID' using loc indexer:
cme.loc[:, 'GST_ActivityID'] = cme['linkedEvents'].apply(lambda x: extract_activityID_from_dict(x))
display(cme.head())

,activityID,startTime,linkedEvents,GST_ActivityID
0,2013-05-01T03:12:00-CME-001,2013-05-01T03:12Z,activityID,None
1,2013-05-03T22:36:00-CME-001,2013-05-03T22:36Z,activityID,None
2,2013-05-09T19:29:00-CME-001,2013-05-09T19:29Z,activityID,None
3,2013-05-13T02:54:00-CME-001,2013-05-13T02:54Z,activityID,None
4,2013-05-13T02:54:00-CME-001,2013-05-13T02:54Z,activityID,None


In [51]:
# Remove rows with missing GST_ActivityID, since we can't assign them to GSTs:
cme = cme[cme['GST_ActivityID'].notna()]

In [53]:
# print out the datatype of each column in this DataFrame:
cme.info()


<class 'pandas.core.frame.DataFrame'>
Index: 0 entries
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype              
---  ------          --------------  -----              
 0   cmeID           0 non-null      object             
 1   startTime_CME   0 non-null      datetime64[ns, UTC]
 2   GST_ActivityID  0 non-null      string             
dtypes: datetime64[ns, UTC](1), object(1), string(1)
memory usage: 0.0+ bytes


In [57]:
# Convert the 'GST_ActivityID' column to string format 
cme['GST_ActivityID'] = cme['GST_ActivityID'].astype('string')

# Convert startTime to datetime format  
cme['startTime_CME'] = pd.to_datetime(cme['startTime_CME'], utc=True)

# Rename startTime to startTime_CME and activityID to cmeID
cme = cme.rename(columns={'activityID': 'cmeID'})

# Drop linkedEvents
if 'linkedEvents' in cme.columns:
    cme = cme.drop(columns=['linkedEvents'])

# Verify that all steps were executed correctly
cme.info()


<class 'pandas.core.frame.DataFrame'>
Index: 0 entries
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype              
---  ------          --------------  -----              
 0   cmeID           0 non-null      object             
 1   startTime_CME   0 non-null      datetime64[ns, UTC]
 2   GST_ActivityID  0 non-null      string             
dtypes: datetime64[ns, UTC](1), object(1), string(1)
memory usage: 0.0+ bytes


In [68]:
# We are only interested in CMEs related to GSTs so keep only rows where the GST_ActivityID column contains 'GST'
# use the method 'contains()' from the str library.  
cme = cme[cme['GST_ActivityID'].str.contains('GST')]

# Do NOT reset the index to keep the original row numbers
# If you wanted to reset the index, you would use: cme = cme.reset_index(drop=True)

cme.head()

,cmeID,startTime_CME,GST_ActivityID


### GST Data

In [69]:
# Set the base URL to NASA's DONKI API:
base_url = "https://api.nasa.gov/DONKI/"

# Set the specifier for Geomagnetic Storms (GST):
GST = "GST"

# Search for GSTs between a begin and end date
startDate = "2013-05-01"
endDate   = "2024-05-01"

# Build URL for GST
query_url_GST = f"{base_url}{GST}?startDate={startDate}&endDate={endDate}&api_key={NASA_API_KEY}"

In [70]:
# Make a "GET" request for the GST URL and store it in a variable named gst_response
gst_response = requests.get(query_url_GST)

In [71]:
# Convert the response variable to json and store it as a variable named gst_json
gst_json = gst_response.json()

In [72]:
# Preview ONLY the first element from the gst_json list you created in JSON format
# Do NOT print out the entire list
# Use json.dumps with argument indent=4 to format data
print(json.dumps(gst_json[0], indent=4))

{
    "gstID": "2013-06-01T01:00:00-GST-001",
    "startTime": "2013-06-01T01:00Z",
    "allKpIndex": [
        {
            "observedTime": "2013-06-01T01:00Z",
            "kpIndex": 6.0,
            "source": "NOAA"
        }
    ],
    "link": "https://webtools.ccmc.gsfc.nasa.gov/DONKI/view/GST/326/-1",
    "linkedEvents": [
        {
            "activityID": "2013-05-31T15:45:00-HSS-001"
        }
    ],
    "submissionTime": "2013-07-15T19:26Z",
    "versionId": 1
}


In [73]:
# Convert gst_json to a Pandas DataFrame  
gst = pd.DataFrame(gst_json)

# Keep only the columns: gstID, startTime, linkedEvents
gst = gst[['gstID', 'startTime', 'linkedEvents']]
gst.head()

,gstID,startTime,linkedEvents
0,2013-06-01T01:00:00-GST-001,2013-06-01T01:00Z,[{'activityID': '2013-05-31T15:45:00-HSS-001'}]
1,2013-06-07T03:00:00-GST-001,2013-06-07T03:00Z,[{'activityID': '2013-06-02T20:24:00-CME-001'}]
2,2013-06-29T03:00:00-GST-001,2013-06-29T03:00Z,None
3,2013-10-02T03:00:00-GST-001,2013-10-02T03:00Z,[{'activityID': '2013-09-29T22:40:00-CME-001'}...
4,2013-12-08T00:00:00-GST-001,2013-12-08T00:00Z,[{'activityID': '2013-12-04T23:12:00-CME-001'}...


In [74]:
# Notice that the linkedEvents column allows us to identify the corresponding CME
# Remove rows with missing 'linkedEvents' since we won't be able to assign these to CME
gst = gst[gst['linkedEvents'].notna()]
gst.head()

,gstID,startTime,linkedEvents
0,2013-06-01T01:00:00-GST-001,2013-06-01T01:00Z,[{'activityID': '2013-05-31T15:45:00-HSS-001'}]
1,2013-06-07T03:00:00-GST-001,2013-06-07T03:00Z,[{'activityID': '2013-06-02T20:24:00-CME-001'}]
3,2013-10-02T03:00:00-GST-001,2013-10-02T03:00Z,[{'activityID': '2013-09-29T22:40:00-CME-001'}...
4,2013-12-08T00:00:00-GST-001,2013-12-08T00:00Z,[{'activityID': '2013-12-04T23:12:00-CME-001'}...
5,2014-02-19T03:00:00-GST-001,2014-02-19T03:00Z,[{'activityID': '2014-02-16T14:15:00-CME-001'}...


In [75]:
# Notice that the linkedEvents sometimes contains multiple events per row
# Use the explode method to ensure that each row is one element. Ensure to reset the index and drop missing values.
gst = gst.explode('linkedEvents').reset_index(drop=True).dropna(subset=['linkedEvents'])
gst.head()

,gstID,startTime,linkedEvents
0,2013-06-01T01:00:00-GST-001,2013-06-01T01:00Z,{'activityID': '2013-05-31T15:45:00-HSS-001'}
1,2013-06-07T03:00:00-GST-001,2013-06-07T03:00Z,{'activityID': '2013-06-02T20:24:00-CME-001'}
2,2013-10-02T03:00:00-GST-001,2013-10-02T03:00Z,{'activityID': '2013-09-29T22:40:00-CME-001'}
3,2013-10-02T03:00:00-GST-001,2013-10-02T03:00Z,{'activityID': '2013-10-02T01:54:00-IPS-001'}
4,2013-10-02T03:00:00-GST-001,2013-10-02T03:00Z,{'activityID': '2013-10-02T02:47:00-MPC-001'}


In [76]:
# Apply the extract_activityID_from_dict function to each row in the 'linkedEvents' column (you can use apply() and a lambda function)
# and create a new column called 'CME_ActivityID' using loc indexer:
gst.loc[:, 'CME_ActivityID'] = gst['linkedEvents'].apply(lambda x: extract_activityID_from_dict(x))
gst.head()
# Remove rows with missing CME_ActivityID, since we can't assign them to CMEs:
gst = gst[gst['CME_ActivityID'].notna()]
gst.head()

,gstID,startTime,linkedEvents,CME_ActivityID
0,2013-06-01T01:00:00-GST-001,2013-06-01T01:00Z,{'activityID': '2013-05-31T15:45:00-HSS-001'},2013-05-31T15:45:00-HSS-001
1,2013-06-07T03:00:00-GST-001,2013-06-07T03:00Z,{'activityID': '2013-06-02T20:24:00-CME-001'},2013-06-02T20:24:00-CME-001
2,2013-10-02T03:00:00-GST-001,2013-10-02T03:00Z,{'activityID': '2013-09-29T22:40:00-CME-001'},2013-09-29T22:40:00-CME-001
3,2013-10-02T03:00:00-GST-001,2013-10-02T03:00Z,{'activityID': '2013-10-02T01:54:00-IPS-001'},2013-10-02T01:54:00-IPS-001
4,2013-10-02T03:00:00-GST-001,2013-10-02T03:00Z,{'activityID': '2013-10-02T02:47:00-MPC-001'},2013-10-02T02:47:00-MPC-001


In [77]:
# Convert the 'CME_ActivityID' column to string format 
gst['CME_ActivityID'] = gst['CME_ActivityID'].astype('string')

# Convert the 'gstID' column to string format 
gst['gstID'] = gst['gstID'].astype('string')

# Convert startTime to datetime format  
gst['startTime'] = pd.to_datetime(gst['startTime'], utc=True)

# Rename startTime to startTime_GST 
gst = gst.rename(columns={'startTime': 'startTime_GST'})

# Drop linkedEvents
gst = gst.drop(columns=['linkedEvents'])

# Verify that all steps were executed correctly
gst.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 205 entries, 0 to 204
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype              
---  ------          --------------  -----              
 0   gstID           205 non-null    string             
 1   startTime_GST   205 non-null    datetime64[ns, UTC]
 2   CME_ActivityID  205 non-null    string             
dtypes: datetime64[ns, UTC](1), string(2)
memory usage: 4.9 KB


In [78]:
# We are only interested in GSTs related to CMEs so keep only rows where the CME_ActivityID column contains 'CME'
# use the method 'contains()' from the str library.  
gst = gst[gst['CME_ActivityID'].str.contains('CME')]
gst.head() 


,gstID,startTime_GST,CME_ActivityID
1,2013-06-07T03:00:00-GST-001,2013-06-07 03:00:00+00:00,2013-06-02T20:24:00-CME-001
2,2013-10-02T03:00:00-GST-001,2013-10-02 03:00:00+00:00,2013-09-29T22:40:00-CME-001
5,2013-12-08T00:00:00-GST-001,2013-12-08 00:00:00+00:00,2013-12-04T23:12:00-CME-001
7,2014-02-19T03:00:00-GST-001,2014-02-19 03:00:00+00:00,2014-02-16T14:15:00-CME-001
9,2014-02-20T03:00:00-GST-001,2014-02-20 03:00:00+00:00,2014-02-18T01:25:00-CME-001


### Merge both datatsets

In [81]:
# Now merge both datasets using 'gstID' and 'CME_ActivityID' for gst and 'GST_ActivityID' and 'cmeID' for cme. Use the 'left_on' and 'right_on' specifiers.
merged_df = pd.merge(gst, cme, 
                    left_on=['gstID', 'CME_ActivityID'], 
                    right_on=['GST_ActivityID', 'cmeID'])

# Set display options for better formatting
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', 1000)        # Wide display
pd.set_option('display.colheader_justify', 'center')  # Center headers

# Use alternative display methods
from IPython.display import display
display(merged_df.head())

,gstID,startTime_GST,CME_ActivityID,cmeID,startTime_CME,GST_ActivityID


In [82]:
# Verify that the new DataFrame has the same number of rows as cme and gst
print("Number of rows in merged_df:", len(merged_df))
print("Number of rows in filtered cme:", len(cme))
print("Number of rows in filtered gst:", len(gst))
merged_df.info()


Number of rows in merged_df: 0
Number of rows in filtered cme: 0
Number of rows in filtered gst: 61
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype              
---  ------          --------------  -----              
 0   gstID           0 non-null      string             
 1   startTime_GST   0 non-null      datetime64[ns, UTC]
 2   CME_ActivityID  0 non-null      string             
 3   cmeID           0 non-null      object             
 4   startTime_CME   0 non-null      datetime64[ns, UTC]
 5   GST_ActivityID  0 non-null      string             
dtypes: datetime64[ns, UTC](2), object(1), string(3)
memory usage: 124.0+ bytes


### Computing the time it takes for a CME to cause a GST

In [83]:
# Compute the time diff between startTime_GST and startTime_CME by creating a new column called `timeDiff`.
merged_df['timeDiff'] = merged_df['startTime_GST'] - merged_df['startTime_CME']
merged_df.head()

,gstID,startTime_GST,CME_ActivityID,cmeID,startTime_CME,GST_ActivityID,timeDiff


In [84]:
# Use describe() to compute the mean and median time 
# that it takes for a CME to cause a GST. 
time_stats = merged_df['timeDiff'].describe()
print(time_stats)


count      0
mean     NaT
std      NaT
min      NaT
25%      NaT
50%      NaT
75%      NaT
max      NaT
Name: timeDiff, dtype: object


### Exporting data in csv format

In [85]:
# Export data to CSV without the index
merged_df.to_csv('cme_gst_data.csv', index=False)
print("Data successfully exported to 'cme_gst_data.csv'")


Data successfully exported to 'cme_gst_data.csv'
